# LangChain tools

Python REPL

In [4]:
from langchain_core.tools import Tool
from langchain_experimental.utilities import PythonREPL

In [6]:
python_repl = PythonREPL()
python_repl.run("print(1+1)")

Python REPL can execute arbitrary code. Use with caution.


'2\n'

Wikipedia

In [10]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.invoke({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


Regular chain

In [21]:
from langchain import OpenAI, PromptTemplate
from langchain.chains import LLMChain

# Initialize LLM
llm = OpenAI(temperature=0)

# Define a prompt template
template = "Translate this text to French:{text}"
prompt = PromptTemplate(input_variables=["text"], template=template)

# Create the chain
t_chain = LLMChain(llm=llm, prompt=prompt)

# Run
a = t_chain.run({"text": "Hello, how are you?"})
print(a)



Bonjour, comment ça va?


Sequential chain

In [22]:
from langchain.chains import SequentialChain
from langchain import LLMChain
from langchain.prompts import PromptTemplate

# Two simple LLMChains
first_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Write a short summary about {topic}."
)
second_prompt = PromptTemplate(
    input_variables=["summary"],
    template="Translate this summary to Spanish: {summary}"
)

chain1 = LLMChain(llm=llm, prompt=first_prompt, output_key="summary")
chain2 = LLMChain(llm=llm, prompt=second_prompt, output_key="translated")

# Build SequentialChain
seq_chain = SequentialChain(
    chains=[chain1, chain2],
    input_variables=["topic"],
    output_variables=["translated"]
)

res = seq_chain.run({"topic": "The benefits of exercise"})
print(res)



El ejercicio tiene numerosos beneficios tanto para la salud física como mental. El ejercicio regular puede mejorar la salud cardiovascular, aumentar la fuerza y flexibilidad muscular, y ayudar a mantener un peso saludable. También libera endorfinas, que pueden mejorar el estado de ánimo y reducir el estrés y la ansiedad. El ejercicio también puede mejorar la función cognitiva y reducir el riesgo de enfermedades crónicas como la diabetes, enfermedades cardíacas y ciertos tipos de cáncer. Además, puede mejorar la calidad del sueño y aumentar los niveles de energía. En general, incorporar el ejercicio en la rutina diaria puede llevar a una vida más saludable y feliz.


MultiPromptChain

In [28]:
from langchain.chains.router.multi_prompt import MultiPromptChain

# For each branch give it a name, a human‑readable description,
# and the template it should run.
prompt_infos = [
    {
        "name": "ingredients",
        "description": "List ingredients for the dish",
        "prompt_template": "List ingredients for {input}.",
    },
    {
        "name": "instructions",
        "description": "Provide cooking steps for the dish",
        "prompt_template": "Provide cooking steps for {input}.",
    },
]

multi_chain = MultiPromptChain.from_prompts(llm, prompt_infos)

# Now just pass a single string in, and the built‑in LLM router
# will pick which branch to run:
print(multi_chain.run("pancakes"))

/Users/boyanangelov/work/oreilly_scalable_products_course_temp/interactive/.venv_anonym/lib/python3.11/site-packages/pydantic/main.py:253: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)




1. In a large mixing bowl, combine 1 1/2 cups of all-purpose flour, 3 1/2 teaspoons of baking powder, 1 teaspoon of salt, and 1 tablespoon of sugar. Mix well.

2. In a separate bowl, beat 1 egg and then add 1 1/4 cups of milk and 3 tablespoons of melted butter. Mix until well combined.

3. Slowly pour the wet ingredients into the dry ingredients, stirring until just combined. Do not overmix, as this can result in tough pancakes.

4. Heat a non-stick griddle or large skillet over medium heat. You can also use a lightly greased pan if you don't have a non-stick surface.

5. Once the pan is hot, use a 1/4 cup measuring cup to scoop the batter onto the pan. Cook for 2-3 minutes, or until bubbles start to form on the surface of the pancake.

6. Use a spatula to flip the pancake and cook for an additional 1-2 minutes, or until both sides are golden brown.

7. Repeat with the remaining batter, adding more butter or oil to the pan as needed.

8. Serve the pancakes warm with your


Router chain

In [35]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate

# 1) your LLM
llm = OpenAI(temperature=0)

# 2) your two “chains” as prompt → llm
prompt_math = PromptTemplate(
    input_variables=["query"],
    template="Solve this math problem: {query}",
)
math_chain = prompt_math | llm

prompt_qa = PromptTemplate(
    input_variables=["query"],
    template="Answer this general knowledge question: {query}",
)
qa_chain = prompt_qa | llm

# 3) your router prompt (just another prompt → llm)
router_prompt = PromptTemplate(
    input_variables=["query"],
    template=(
        "You are a router.\n"
        "Decide whether the user query is a MATH problem or a GENERAL knowledge question.\n"
        "Respond exactly with 'MATH' or 'GENERAL'.\n\n"
        "User query: {query}"
    ),
)
router_chain = router_prompt | llm

# 4) helper to run the right chain
def router_run(query: str) -> str:
    choice = router_chain.invoke({"query": query}).strip().upper()
    if choice == "MATH":
        return math_chain.invoke({"query": query})
    # default to QA
    return qa_chain.invoke({"query": query})

# 5) now it just works:
print(router_run("What is 13 * 7?"))                 # → uses math_chain
print(router_run("Who wrote 'Pride and Prejudice'?"))  # → uses qa_chain



13 * 7 = 91


Jane Austen
